In [65]:
import psycopg2
import pandas as pd
from datetime import datetime, timedelta, date

In [69]:
import yfinance as yf

ticker = yf.Ticker("AAPL")

In [3]:
ticker

yfinance.Ticker object <AAPL>

In [4]:
print(dir(ticker))

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_analysis', '_data', '_download_options', '_earnings', '_earnings_dates', '_expirations', '_fast_info', '_fetch_ticker_tz', '_financials', '_fundamentals', '_funds_data', '_get_earnings_dates_using_scrape', '_get_earnings_dates_using_screener', '_get_ticker_tz', '_holders', '_isin', '_lazy_load_price_history', '_message_handler', '_news', '_options2df', '_price_history', '_quote', '_shares', '_tz', '_underlying', 'actions', 'analyst_price_targets', 'balance_sheet', 'balancesheet', 'calendar', 'capital_gains', 'cash_flow', 'cashflow', 'dividends', 'earnings', 'earnings_dates', 'earnings_estimate', 'earnings_history', 'eps_revisions

In [5]:
ticker.analyst_price_targets

{'current': 333.74,
 'high': 400.0,
 'low': 215.0,
 'mean': 315.78604,
 'median': 315.0}

In [59]:
history = ticker.history(start=datetime.now()-timedelta(days=10), interval="1d", auto_adjust=False)
history

,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits
Date,,,,,,,,
2026-07-08 00:00:00-04:00,311.910004,314.820007,307.049988,313.390015,313.390015,41323500,0.0,0.0
2026-07-09 00:00:00-04:00,310.510010,316.529999,308.160004,316.220001,316.220001,48124500,0.0,0.0
2026-07-10 00:00:00-04:00,314.720001,316.910004,312.170013,315.320007,315.320007,34132300,0.0,0.0
2026-07-13 00:00:00-04:00,317.019989,323.450012,315.779999,317.309998,317.309998,43257800,0.0,0.0
2026-07-14 00:00:00-04:00,313.760010,316.190002,311.910004,314.859985,314.859985,36336800,0.0,0.0
2026-07-15 00:00:00-04:00,317.619995,328.730011,317.320007,327.500000,327.500000,60957600,0.0,0.0
2026-07-16 00:00:00-04:00,328.010010,334.679993,326.790009,333.260010,333.260010,62970600,0.0,0.0
2026-07-17 00:00:00-04:00,331.980011,334.980011,329.000610,333.739990,333.739990,63325386,0.0,0.0


In [58]:
rows = []

for date, row in history.iterrows():
    rows.append((
        ticker,
        date.date(),
        row["Open"],
        row["High"],
        row["Low"],
        row["Close"],
        row["Volume"]
    ))
rows

[(yfinance.Ticker object <AAPL>,
  datetime.date(2026, 7, 8),
  np.float64(311.9100036621094),
  np.float64(314.82000732421875),
  np.float64(307.04998779296875),
  np.float64(313.3900146484375),
  np.float64(41323500.0)),
 (yfinance.Ticker object <AAPL>,
  datetime.date(2026, 7, 9),
  np.float64(310.510009765625),
  np.float64(316.5299987792969),
  np.float64(308.1600036621094),
  np.float64(316.2200012207031),
  np.float64(48124500.0)),
 (yfinance.Ticker object <AAPL>,
  datetime.date(2026, 7, 10),
  np.float64(314.7200012207031),
  np.float64(316.9100036621094),
  np.float64(312.1700134277344),
  np.float64(315.32000732421875),
  np.float64(34132300.0)),
 (yfinance.Ticker object <AAPL>,
  datetime.date(2026, 7, 13),
  np.float64(317.0199890136719),
  np.float64(323.45001220703125),
  np.float64(315.7799987792969),
  np.float64(317.30999755859375),
  np.float64(43257800.0)),
 (yfinance.Ticker object <AAPL>,
  datetime.date(2026, 7, 14),
  np.float64(313.760009765625),
  np.float64(31

In [16]:
## Connect to PostGres database
conn = psycopg2.connect(
    host="localhost",
    database="alphalab",
    user="postgres",
    password="alphalab",
    port="5432"
)
cursor = conn.cursor()

In [39]:
query = """
        SELECT MAX(date)
        FROM daily_prices
        WHERE ticker = %s
        """
cursor.execute(query, ('AAPL',))

In [40]:
latest_date = cursor.fetchall()[0][0]
latest_date

datetime.date(2026, 7, 17)

In [41]:
datetime.now()

datetime.datetime(2026, 7, 17, 21, 19, 20, 982163)

In [36]:
query = """
INSERT INTO daily_prices (
    ticker,
    date,
    open,
    high,
    low,
    close,
    adj_close,
    volume,
    last_updated
)
VALUES (
    %s,%s,%s,%s,%s,
    %s,%s,%s,%s
)
ON CONFLICT (ticker, date)
DO NOTHING
"""

data = (
    'AAPL',
    datetime.now(),
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0,
    datetime.now()
)

cursor.execute(query, data)
conn.commit()

In [61]:
cursor.close()
conn.close()

In [23]:
#conn.rollback()

In [62]:
ticker.dividends

Date
1987-05-11 09:30:00-04:00    0.000536
1987-08-10 09:30:00-04:00    0.000536
1987-11-17 09:30:00-05:00    0.000714
1988-02-12 09:30:00-05:00    0.000714
1988-05-16 09:30:00-04:00    0.000714
                               ...   
2025-05-12 09:30:00-04:00    0.260000
2025-08-11 09:30:00-04:00    0.260000
2025-11-10 09:30:00-05:00    0.260000
2026-02-09 09:30:00-05:00    0.260000
2026-05-11 09:30:00-04:00    0.270000
Name: Dividends, Length: 91, dtype: float64

In [8]:
ticker.get_info()

{'address1': 'One Apple Park Way',
 'city': 'Cupertino',
 'state': 'CA',
 'zip': '95014',
 'country': 'United States',
 'phone': '(408) 996-1010',
 'website': 'https://www.apple.com',
 'industry': 'Consumer Electronics',
 'industryKey': 'consumer-electronics',
 'industryDisp': 'Consumer Electronics',
 'sector': 'Technology',
 'sectorKey': 'technology',
 'sectorDisp': 'Technology',
 'longBusinessSummary': 'Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accessories worldwide. The company offers iPhone, a line of smartphones; Mac, a line of personal computers; iPad, a line of multi-purpose tablets; and wearables, home, and accessories comprising AirPods, Apple Vision Pro, Apple TV, Apple Watch, Beats products, and HomePod, as well as Apple branded and third-party accessories. It also provides AppleCare support and cloud services; and operates various platforms, including the App Store that allows customers to discover and download ap

In [9]:
info = ticker.info
fast = ticker.fast_info

stock = {
    "ticker": "AAPL",
    "company_name": info.get("longName"),
    "exchange": info.get("exchange"),
    "currency": fast.get("currency"),
    "sector": info.get("sector"),
    "industry": info.get("industry"),
    "country": info.get("country"),
    "asset_type": fast.get("asset_type"),
    "timezone": fast.get("timezone"),
    "is_active": True
}

print(stock)

{'ticker': 'AAPL', 'company_name': 'Apple Inc.', 'exchange': 'NMS', 'currency': 'USD', 'sector': 'Technology', 'industry': 'Consumer Electronics', 'country': 'United States', 'asset_type': None, 'timezone': 'America/New_York', 'is_active': True}


In [10]:
yf.tickers

<module 'yfinance.tickers' from 'C:\\Users\\felip\\AppData\\Local\\Programs\\Python\\Python312\\Lib\\site-packages\\yfinance\\tickers.py'>